Load Env

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env.local")

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")


Download Dataset (Optional, I already provide some pic in discord)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)  
project = rf.workspace("licenseplatethai").project("thailand-license-plate-recognition-cp5pl")
dataset = project.version(1).download("yolov8")

Easy OCR init

In [ ]:
import easyocr
from PIL import Image
import numpy as np

reader = easyocr.Reader(['th'], gpu=False)
ALLOWLIST = "".join(chr(c) for c in range(0x0E01, 0x0E3B)) + "0123456789"

Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

You can tune parameter here

In [ ]:
def run_easyocr(img):
    """Run EasyOCR on a PIL image."""
    arr = np.array(img)
    results = reader.readtext(
        arr,
        detail=1,
        paragraph=False,
        contrast_ths=0.1,        
        adjust_contrast=1.5,   
        text_threshold=0.5,     
        low_text=0.2,          
        link_threshold=0.3,  
        allowlist=ALLOWLIST,
        decoder="beamsearch",
        rotation_info=[0, 90, -90],
        width_ths=0.5,        
        height_ths=0.5
    )
    return results

Read Image from Train set where we download

In [14]:
import os
from PIL import Image
import glob

# หารูปภาพในโฟลเดอร์
def find_images(dataset_path):
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']
    images = []
    
    for ext in image_extensions:
        images.extend(glob.glob(os.path.join(dataset_path, "**", ext), recursive=True))
    
    return images

# In case you download dataset from Roboflow
# images = find_images("./Thailand-License-Plate-Recognition-1/train/images")

# In case you use my provided images
images = find_images("./images")

Test EasyOCR with our first image

In [17]:
import os
from PIL import Image

# เช็ครูปภาพที่พบ
if images:
    print(f"รูปแรกที่พบ: {images[0]}")
    print(f"ชื่อไฟล์: {os.path.basename(images[0])}")
    print(f"ขนาดไฟล์: {os.path.getsize(images[0])} bytes")
    
    # ดูขนาดรูป
    img = Image.open(images[0])
    print(f"ขนาดรูป: {img.size} (width x height)")
    print(f"โหมดสี: {img.mode}")
    
    # ทดสอบ OCR
    img_rgb = img.convert("RGB")
    results = run_easyocr(img_rgb)
    print(f"\nผลลัพธ์ OCR:")
    for bbox, text, conf in results:
        print(f"  - ข้อความ: '{text}' | ความเชื่อมั่น: {conf:.3f}")
else:
    print("ไม่พบรูปภาพในโฟลเดอร์")

รูปแรกที่พบ: ./images\image01.jpg
ชื่อไฟล์: image01.jpg
ขนาดไฟล์: 3934 bytes
ขนาดรูป: (199, 82) (width x height)
โหมดสี: RGB

ผลลัพธ์ OCR:
  - ข้อความ: '1ก58107' | ความเชื่อมั่น: 0.868
  - ข้อความ: 'กรงทพมทานคร' | ความเชื่อมั่น: 0.553


Google cloud vision api instead of EasyOCR

In [21]:
import os
from google.cloud import vision
from google.oauth2 import service_account

def google_ocr(image_path, credentials_path=None):
    try:
        if credentials_path:
            credentials = service_account.Credentials.from_service_account_file(credentials_path)
            client = vision.ImageAnnotatorClient(credentials=credentials)
        else:
            # ใช้ environment variable GOOGLE_APPLICATION_CREDENTIALS
            client = vision.ImageAnnotatorClient()
        
        with open(image_path, 'rb') as image_file:
            content = image_file.read()
        
        image = vision.Image(content=content)
        response = client.text_detection(image=image)
        
        # เช็ค error
        if response.error.message:
            raise Exception(f'Google Vision API Error: {response.error.message}')
        
        texts = response.text_annotations
        if texts:
            return texts[0].description.strip()
        return ""
        
    except Exception as e:
        print(f"Error: {e}")
        return ""

In [28]:
# ใช้งาน
result = google_ocr('images/image06.jpg', 'ggcloud-key.json')
print(f"Google Vision Result: {result}")

Google Vision Result: กย 8756
เชียงใหม่ ล


In [31]:
import os
import re
from difflib import SequenceMatcher

def read_label_file(label_path):
    """อ่านไฟล์ label (หลายบรรทัด)"""
    try:
        with open(label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            # รวมทุกบรรทัดเป็นข้อความเดียว คั่นด้วยช่องว่าง
            return ' '.join(line.strip() for line in lines if line.strip())
    except FileNotFoundError:
        return None
    
def calculate_similarity(ocr_result, true_label):
    """คำนวณความคล้ายกันเป็นเปอร์เซ็นต์"""
    if not ocr_result and not true_label:
        return 100.0  # ทั้งคู่ว่าง
    if not ocr_result or not true_label:
        return 0.0    # ฝั่งใดฝั่งหนึ่งว่าง
    
    # ทำความสะอาดข้อความ
    clean_ocr = clean_text(ocr_result)
    clean_label = clean_text(true_label)
    
    # คำนวณความคล้ายกันด้วย SequenceMatcher
    similarity = SequenceMatcher(None, clean_ocr.lower(), clean_label.lower()).ratio()
    
    return similarity * 100

def clean_text(text):
    """ทำความสะอาดข้อความ - เอาตัวแปลกๆ ออก"""
    if not text:
        return ""
    
    # เอาช่องว่างและบรรทัดใหม่ออก
    text = re.sub(r'\s+', ' ', text.strip())
    
    # เก็บแค่ตัวอักษรไทย ตัวเลข และตัวอักษรอังกฤษ
    # Unicode range สำหรับภาษาไทย: \u0E00-\u0E7F
    cleaned = re.sub(r'[^\u0E00-\u0E7F0-9A-Za-z\s]', '', text)
    
    # เอาช่องว่างซ้ำออก
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    return cleaned

def calculate_character_accuracy(ocr_result, true_label):
    """คำนวณความแม่นยำระดับตัวอักษร"""
    clean_ocr = clean_text(ocr_result).replace(' ', '')
    clean_label = clean_text(true_label).replace(' ', '')
    
    if not clean_label:
        return 0.0
    
    # นับตัวอักษรที่ตรงกัน
    correct_chars = 0
    total_chars = len(clean_label)
    
    for i in range(min(len(clean_ocr), len(clean_label))):
        if clean_ocr[i] == clean_label[i]:
            correct_chars += 1
    
    return (correct_chars / total_chars) * 100

def compare_ocr_result(image_path, credentials_path=None):
    """เปรียบเทียบผลลัพธ์ OCR กับ label จริง (แบบกระชับ)"""
    
    # สร้าง path ของ label file
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    label_path = f"labels/label_{image_name}.txt"
    
    # อ่าน label จริง
    true_label = read_label_file(label_path)
    if true_label is None:
        print(f"ไม่พบไฟล์ label: {label_path}")
        return
    
    # ทำ OCR
    ocr_result = google_ocr(image_path, credentials_path)
    
    # คำนวณความแม่นยำ
    char_accuracy = calculate_character_accuracy(ocr_result, true_label)
    
    # แสดงผลลัพธ์แบบกระชับ
    clean_ocr = clean_text(ocr_result)
    print(f"{image_name}: '{true_label}' vs '{clean_ocr}' - {char_accuracy:.1f}%")
    
    return {
        'image': image_name,
        'true_label': true_label,
        'ocr_result': clean_ocr,
        'accuracy': char_accuracy
    }
# ทดสอบ
result = compare_ocr_result('images/image02.jpg', 'ggcloud-key.json')

image02: 'กง 4275 พังงา' vs 'กง 4275 พังงา' - 100.0%
